# 01 · Score the detectors

Loads the in-the-wild jailbreak dataset (Shen et al., CCS 2024), removes duplicates and scores every prompt with four detectors.

In [ ]:
!pip install -q datasets

In [ ]:
import os
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/jailbreak_project"
os.makedirs(SAVE_DIR, exist_ok=True)

## 1. Load and deduplicate

Release `2023_12_25`: jailbreak split = 1, regular split = 0. A prompt found in both splits is kept only as jailbreak. The row order defines `uid`.

In [ ]:
REPO = "TrustAIRLab/in-the-wild-jailbreak-prompts"
jb = load_dataset(REPO, "jailbreak_2023_12_25", split="train").to_pandas()
reg = load_dataset(REPO, "regular_2023_12_25", split="train").to_pandas()


def dedup(df):
    df = df.copy()
    df["prompt"] = df["prompt"].astype(str).str.strip()
    df = df[df["prompt"] != ""]
    return df.drop_duplicates(subset="prompt").reset_index(drop=True)


jb, reg = dedup(jb), dedup(reg)
reg = reg[~reg["prompt"].isin(set(jb["prompt"]))].reset_index(drop=True)
jb["label"], reg["label"] = 1, 0
data = pd.concat([jb, reg], ignore_index=True)
print(len(data), "prompts:", len(jb), "jailbreak,", len(reg), "regular")

## 2. Score

Prompts are split into 510-token chunks and a prompt's score is its highest chunk score.

In [ ]:
DETECTORS = {
    "protectai": "protectai/deberta-v3-base-prompt-injection-v2",
    "pg2_22m": "meta-llama/Llama-Prompt-Guard-2-22M",
    "pg2_86m": "meta-llama/Llama-Prompt-Guard-2-86M",
    "piguard": "leolee99/PIGuard",
}
ATTACK = {"INJECTION", "JAILBREAK", "MALICIOUS", "LABEL_1"}   # attack labels of the four models


def score(model_id, texts, trust=False):
    clf = pipeline("text-classification", model=model_id, device=0, top_k=None,
                   truncation=True, max_length=512, trust_remote_code=trust)
    tok = clf.tokenizer

    # split each prompt into 510-token chunks (512 with the special tokens)
    chunks, owner = [], []
    for i, t in enumerate(texts):
        ids = tok(t, add_special_tokens=False)["input_ids"]
        parts = [t] if len(ids) <= 510 else [tok.decode(ids[k:k + 510], skip_special_tokens=True)
                                             for k in range(0, len(ids), 510)]
        chunks += parts
        owner += [i] * len(parts)

    # score the chunks, shortest first so that each batch has similar lengths
    order = np.argsort([len(c) for c in chunks])
    results = clf([chunks[j] for j in order], batch_size=32)
    chunk_score = np.zeros(len(chunks))
    for j, res in zip(order, results):
        chunk_score[j] = sum(r["score"] for r in res if r["label"].upper() in ATTACK)

    # a prompt's score is its highest chunk score
    prompt_score = np.zeros(len(texts))
    np.maximum.at(prompt_score, np.array(owner), chunk_score)
    return prompt_score

In [ ]:
texts = data["prompt"].tolist()
for name, model_id in DETECTORS.items():
    path = f"{SAVE_DIR}/scores_{name}.npy"
    if not os.path.exists(path):
        np.save(path, score(model_id, texts, trust=(name == "piguard")))
    data[name] = np.load(path)

## 3. Save

`scores_rq1.csv` (`uid`, original tag and the four scores) is published as `data/scores_rq1.csv`. `scores_step2.csv` also keeps the prompt text, for notebook 02.

In [ ]:
data.to_csv(f"{SAVE_DIR}/scores_step2.csv", index=False)

pub = data[["label"] + list(DETECTORS)].copy()
pub.insert(0, "uid", range(len(pub)))
pub.to_csv(f"{SAVE_DIR}/scores_rq1.csv", index=False)